In [6]:
import random
import numpy as np
import torch
import os

def set_seed(seed=42):
    # Python random
    random.seed(seed)

    # Numpy
    np.random.seed(seed)

    # PyTorch CPU
    torch.manual_seed(seed)

    # PyTorch GPU
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Ensures deterministic behavior
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # For hash-based ops
    os.environ["PYTHONHASHSEED"] = str(seed)

# Set seed
set_seed(42)


In [9]:

import pandas as pd
import faiss

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# HuggingFace
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer
)

# PyTorch utilities
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn

# Device setup (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load tokenizer once (used for all datasets)
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

In [8]:
!pip install transformers datasets faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 64.3 MB/s eta 0:00:00:00:0100:01


In [49]:

# Tokenization Function
# Converts raw text to input_ids and attention_mask

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )


# Extract CLS embeddings from RoBERTa encoder

def extract_embeddings(model, dataset):
    model.eval()
    loader = DataLoader(dataset, batch_size=32)
    embeddings = []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        with torch.no_grad():
            outputs = model.roberta(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        # CLS token representation
        cls = outputs.last_hidden_state[:, 0, :]
        embeddings.append(cls.cpu())

    return torch.cat(embeddings).numpy()
 


In [ ]:
# Load COVID dataset
# Download datasets from the links provided in README.md
# and update the paths below accordingly
covid_train = pd.read_csv("/path/to/your/dataset/")
covid_test = pd.read_csv("/path/to/your/dataset/")

covid_train["label"] = covid_train["label"].map({"fake":0,"real":1})
covid_test["label"] = covid_test["label"].map({"fake":0,"real":1})

# Convert to HF dataset
covid_train_dataset = Dataset.from_dict({
    "text": covid_train["tweet"].values,
    "label": covid_train["label"].values
}).map(tokenize_batch, batched=True)

covid_test_dataset = Dataset.from_dict({
    "text": covid_test["tweet"].values,
    "label": covid_test["label"].values
}).map(tokenize_batch, batched=True)

covid_train_dataset.set_format("torch", columns=["input_ids","attention_mask","label"])
covid_test_dataset.set_format("torch", columns=["input_ids","attention_mask","label"])


Map:   0%|          | 0/6420 [00:00<?, ? examples/s]

Map:   0%|          | 0/2140 [00:00<?, ? examples/s]

## Cross-Validating ISOT-Initialized RoBERTa on COVID

In [51]:

# Load ISOT trained checkpoint
covid_model = RobertaForSequenceClassification.from_pretrained(
    "./isot_proper_pretraining/checkpoint-5799"  
).to(device)

# Freeze all layers
for p in covid_model.roberta.parameters():
    p.requires_grad = False

# Unfreeze last 3 transformer layers
for layer in covid_model.roberta.encoder.layer[-3:]:
    for p in layer.parameters():
        p.requires_grad = True

# Unfreeze classifier
for p in covid_model.classifier.parameters():
    p.requires_grad = True

covid_args = TrainingArguments(
    output_dir="./covid_adapt",
    num_train_epochs=5,
    learning_rate=1e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    max_grad_norm=1.0,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="no",
    seed=42,
    fp16=True,
    report_to="none"
)



covid_trainer = Trainer(
    model=covid_model,
    args=covid_args,
    train_dataset=covid_train_dataset,
    eval_dataset=covid_test_dataset,
)

covid_trainer.train()

covid_preds = covid_trainer.predict(covid_test_dataset)

isot_covid_acc = accuracy_score(
    covid_preds.label_ids,
    covid_preds.predictions.argmax(axis=1)
)

print("ISOT-Initialized RoBERTa COVID Accuracy:", isot_covid_acc)

Epoch,Training Loss,Validation Loss
1,No log,0.562916
2,1.326600,0.566787
3,0.561300,0.554688
4,0.535500,0.539595
5,0.516000,0.528965


ISOT-Initialized RoBERTa COVID Accuracy: 0.7537383177570094


In [ ]:
#COVID test dataset for final unbiased evaluation
# Download datasets from the links provided in README.md
# and update the paths below accordingly
covid_val = pd.read_csv("/path/to/your/dataset/")

# Convert labels to numeric
covid_val["label"] = covid_val["label"].map({
    "fake": 0,
    "real": 1
})


# Convert COVID validation pandas DataFrame → HuggingFace Dataset

from datasets import Dataset
covid_val_dataset = Dataset.from_dict({

    "text": covid_val["tweet"].values,

    "labels": covid_val["label"].values

})


# Tokenize COVID validation dataset using RoBERTa tokenizer

covid_val_dataset = covid_val_dataset.map(
    tokenize_batch,
    batched=True
)

# Convert COVID validation dataset into PyTorch tensor format
covid_val_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)


Map:   0%|          | 0/2140 [00:00<?, ? examples/s]

In [53]:

from sklearn.metrics import accuracy_score
import numpy as np

# Get predictions
covid_val_preds = covid_trainer.predict(covid_val_dataset)

# Convert logits → predicted labels
predicted_labels = np.argmax(covid_val_preds.predictions, axis=1)

# True labels
true_labels = covid_val_preds.label_ids

# Accuracy
test_accuracy = accuracy_score(true_labels, predicted_labels)

print("COVID Test Accuracy:", test_accuracy)

COVID Test Accuracy: 0.7453271028037384


# gossipcop on isot initialized roberta

In [ ]:

# GossipCop Train / Validation / Test split + ISOT adaptation
# ============================================================

import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from datasets import Dataset
from transformers import RobertaForSequenceClassification, TrainingArguments, Trainer


# ============================================================
# 1. Load and clean GossipCop dataset
# Download datasets from the links provided in README.md
# and update the paths below accordingly
# ============================================================

gossip_df = pd.read_csv("/path/to/your/dataset/")

# Convert labels
gossip_df["label"] = gossip_df["label"].map({"fake": 0, "real": 1})

# Remove missing values
gossip_df = gossip_df.dropna(subset=["text", "label"])

print("Total samples:", len(gossip_df))


# ============================================================
# 2. Proper Train / Validation / Test split
# ============================================================

# Train 80%, Temp 20%
gossip_train_df, gossip_temp_df = train_test_split(
    gossip_df,
    test_size=0.2,
    stratify=gossip_df["label"],
    random_state=42
)

# Validation 10%, Test 10%
gossip_validation_df, gossip_test_df = train_test_split(
    gossip_temp_df,
    test_size=0.5,
    stratify=gossip_temp_df["label"],
    random_state=42
)

print("\nSplit sizes:")
print("Train:", len(gossip_train_df))
print("Validation:", len(gossip_validation_df))
print("Test:", len(gossip_test_df))


# ============================================================
# 3. Convert to HuggingFace Dataset
# ============================================================

gossip_train_dataset = Dataset.from_dict({
    "text": gossip_train_df["text"].values,
    "label": gossip_train_df["label"].values
}).map(tokenize_batch, batched=True)

gossip_validation_dataset = Dataset.from_dict({
    "text": gossip_validation_df["text"].values,
    "label": gossip_validation_df["label"].values
}).map(tokenize_batch, batched=True)

gossip_test_dataset = Dataset.from_dict({
    "text": gossip_test_df["text"].values,
    "label": gossip_test_df["label"].values
}).map(tokenize_batch, batched=True)


# Set torch format
gossip_train_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"]
)

gossip_validation_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"]
)

gossip_test_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"]
)


# ============================================================
# 4. Load ISOT-initialized RoBERTa
# ============================================================

gossip_model = RobertaForSequenceClassification.from_pretrained(
    "./isot_proper_pretraining/checkpoint-5799"
).to(device)


# ============================================================
# 5. Freeze / Unfreeze layers
# ============================================================

# Freeze all encoder layers
for param in gossip_model.roberta.parameters():
    param.requires_grad = False


# Unfreeze last 3 layers
for layer in gossip_model.roberta.encoder.layer[-3:]:
    for param in layer.parameters():
        param.requires_grad = True


# Always unfreeze classifier
for param in gossip_model.classifier.parameters():
    param.requires_grad = True


# ============================================================
# 6. Training Arguments
# ============================================================

gossip_args = TrainingArguments(

    output_dir="./gossip_adapt",

    num_train_epochs=5,

    learning_rate=1e-5,

    warmup_ratio=0.1,

    weight_decay=0.01,

    max_grad_norm=1.0,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    eval_strategy="epoch",

    save_strategy="no",

    seed=42,

    fp16=True,

    report_to="none"
)


# ============================================================
# 7. Trainer
# ============================================================

gossip_trainer = Trainer(

    model=gossip_model,

    args=gossip_args,

    train_dataset=gossip_train_dataset,

    eval_dataset=gossip_validation_dataset
)


# ============================================================
# 8. Train model
# ============================================================

print("\nTraining GossipCop adaptation...")

gossip_trainer.train()


# ============================================================
# 9. Validation performance
# ============================================================

validation_results = gossip_trainer.evaluate(
    eval_dataset=gossip_validation_dataset
)

print("\nValidation Results:", validation_results)


# ============================================================
# 10. Final unbiased Test evaluation
# ============================================================

gossip_test_predictions = gossip_trainer.predict(
    gossip_test_dataset
)

gossip_test_preds = np.argmax(
    gossip_test_predictions.predictions,
    axis=1
)

gossip_test_acc = accuracy_score(
    gossip_test_predictions.label_ids,
    gossip_test_preds
)

print("\nFinal GossipCop Test Accuracy:", gossip_test_acc)

print("\nClassification Report:")

print(
    classification_report(
        gossip_test_predictions.label_ids,
        gossip_test_preds,
        target_names=["Fake", "Real"]
    )
)

Total samples: 20007

Split sizes:
Train: 16005
Validation: 2001
Test: 2001


Map:   0%|          | 0/16005 [00:00<?, ? examples/s]

Map:   0%|          | 0/2001 [00:00<?, ? examples/s]

Map:   0%|          | 0/2001 [00:00<?, ? examples/s]


Training GossipCop adaptation...


Epoch,Training Loss,Validation Loss
1,0.552800,0.560864
2,0.533000,0.499369
3,0.493700,0.470257
4,0.461900,0.479887
5,0.459500,0.468333



Validation Results: {'eval_loss': 0.4683333933353424, 'eval_runtime': 13.8765, 'eval_samples_per_second': 144.2, 'eval_steps_per_second': 9.08, 'epoch': 5.0}

Final GossipCop Test Accuracy: 0.8085957021489255

Classification Report:
              precision    recall  f1-score   support

        Fake       0.76      0.30      0.43       479
        Real       0.81      0.97      0.89      1522

    accuracy                           0.81      2001
   macro avg       0.78      0.63      0.66      2001
weighted avg       0.80      0.81      0.78      2001



# Covid dataset trained on vanilla roberta-base

In [40]:
from transformers import RobertaForSequenceClassification, TrainingArguments, Trainer

vanilla_covid_model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
).to(device)


vanilla_covid_args = TrainingArguments(
    output_dir="./vanilla_covid",
    num_train_epochs=3,
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="no",
    seed=42,
    fp16=True,
    report_to="none"
)

vanilla_covid_trainer = Trainer(
    model=vanilla_covid_model,
    args=vanilla_covid_args,
    train_dataset=covid_train_dataset,
    eval_dataset=covid_test_dataset
)

vanilla_covid_trainer.train()

from sklearn.metrics import accuracy_score

covid_preds = vanilla_covid_trainer.predict(covid_test_dataset)

vanilla_covid_acc = accuracy_score(
    covid_preds.label_ids,
    covid_preds.predictions.argmax(axis=1)
)

print("Vanilla RoBERTa COVID Accuracy:", vanilla_covid_acc)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,No log,0.188181
2,0.215600,0.210032
3,0.085400,0.150478


Vanilla RoBERTa COVID Accuracy: 0.969626168224299


In [41]:


results = pd.DataFrame({
    "Dataset": [
        "COVID", "COVID", "COVID",
        "GossipCop", "GossipCop", "GossipCop"
    ],
    "Model": [
        "Vanilla RoBERTa",
        "ISOT-Initialized RoBERTa",
        "Retrieval + Fusion",
        "Vanilla RoBERTa",
        "ISOT-Initialized RoBERTa",
        "Retrieval + Fusion"
    ],
    "Accuracy": [
        vanilla_covid_acc,
        isot_covid_acc,
        covid_fusion_acc,
        vanilla_gossip_acc,
        isot_gossip_acc,
        gossip_fusion_acc
    ]
})

print(results)



NameError: name 'covid_fusion_acc' is not defined

# gossipcop dataset trained on vanilla Robert-base

In [42]:
#Initialize Vanilla Model
vanilla_gossip_model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
).to(device)
#Training Arguments
vanilla_gossip_args = TrainingArguments(
    output_dir="./vanilla_gossip",
    num_train_epochs=3,
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="no",
    seed=42,
    fp16=True,
    report_to="none"
)

vanilla_gossip_trainer = Trainer(
    model=vanilla_gossip_model,
    args=vanilla_gossip_args,
    train_dataset=gossip_train_dataset,
    eval_dataset=gossip_test_dataset
)
vanilla_gossip_trainer.train()
# Evaluate
gossip_preds = vanilla_gossip_trainer.predict(gossip_test_dataset)

vanilla_gossip_acc = accuracy_score(
    gossip_preds.label_ids,
    gossip_preds.predictions.argmax(axis=1)
)

print("Vanilla RoBERTa Gossip Accuracy:", vanilla_gossip_acc)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,0.352500,0.347175
2,0.303700,0.356793
3,0.268200,0.358863


Vanilla RoBERTa Gossip Accuracy: 0.8630684657671165


# welfake vanilla

In [ ]:
import pandas as pd
# Download datasets from the links provided in README.md
# and update the paths below accordingly
welfake = pd.read_csv(
    "/path/to/your/dataset/"
)

print("Dataset shape:", welfake.shape)
print(welfake.columns)
print(welfake.head())                                                                      

# Clean WELFake dataset, remove invalid rows

# Keep only text and label columns
welfake = welfake[["text", "label"]]

# Drop rows where text is missing
welfake = welfake.dropna(subset=["text"])

# Drop rows where label is missing
welfake = welfake.dropna(subset=["label"])

# Convert label to integer
welfake["label"] = welfake["label"].astype(int)

# Reset index (recommended after dropping rows)
welfake = welfake.reset_index(drop=True)

# Rename text column to content (for compatibility with your pipeline)
welfake = welfake.rename(columns={"text": "content"})

# Check label distribution
print("Label distribution:")
print(welfake["label"].value_counts())

# Check total rows
print("\nTotal rows after cleaning:", len(welfake))

Dataset shape: (72134, 4)
Index(['Unnamed: 0', 'title', 'text', 'label'], dtype='object')
   Unnamed: 0                                              title  \
0           0  LAW ENFORCEMENT ON HIGH ALERT Following Threat...   
1           1                                                NaN   
2           2  UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...   
3           3  Bobby Jindal, raised Hindu, uses story of Chri...   
4           4  SATAN 2: Russia unvelis an image of its terrif...   

                                                text  label  
0  No comment is expected from Barack Obama Membe...      1  
1     Did they post their votes for Hillary already?      1  
2   Now, most of the demonstrators gathered last ...      1  
3  A dozen politically active pastors came here f...      0  
4  The RS-28 Sarmat missile, dubbed Satan 2, will...      1  
Label distribution:
label
1    37067
0    35028
Name: count, dtype: int64

Total rows after cleaning: 72095


In [44]:
welfake_train, welfake_temp = train_test_split(

    welfake,
    test_size=0.3,
    stratify=welfake["label"],
    random_state=42
)

# Second split: test (15%) and validation (15%)
welfake_test, welfake_validation = train_test_split(

    welfake_temp,
    test_size=0.5,
    stratify=welfake_temp["label"],
    random_state=42
)

print("Train:", len(welfake_train))
print("Test:", len(welfake_test))
print("Validation:", len(welfake_validation))

Train: 50466
Test: 10814
Validation: 10815


In [45]:

welfake_train_dataset = Dataset.from_dict({

    "text": welfake_train["content"].values,
    "labels": welfake_train["label"].values
})

welfake_test_dataset = Dataset.from_dict({

    "text": welfake_test["content"].values,
    "labels": welfake_test["label"].values
})

welfake_validation_dataset = Dataset.from_dict({

    "text": welfake_validation["content"].values,
    "labels": welfake_validation["label"].values
})

In [46]:
# Tokenize WELFake datasets

welfake_train_dataset = welfake_train_dataset.map(
    tokenize_batch,
    batched=True
)

welfake_test_dataset = welfake_test_dataset.map(
    tokenize_batch,
    batched=True
)

welfake_validation_dataset = welfake_validation_dataset.map(
    tokenize_batch,
    batched=True
)
# Convert to PyTorch format

welfake_train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

welfake_test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

welfake_validation_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

Map:   0%|          | 0/50466 [00:00<?, ? examples/s]

Map:   0%|          | 0/10814 [00:00<?, ? examples/s]

Map:   0%|          | 0/10815 [00:00<?, ? examples/s]

In [48]:
# Train Vanilla RoBERTa on WELFake and calculate accuracy


import numpy as np
import torch
from sklearn.metrics import accuracy_score, classification_report

from transformers import (
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer
)

# ============================================================
# 1. Load Vanilla RoBERTa
# ============================================================

welfake_vanilla_model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
).to(device)


# ============================================================
# 2. Training Arguments
# ============================================================

welfake_vanilla_args = TrainingArguments(

    output_dir="./welfake_vanilla_results",

    num_train_epochs=3,

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    eval_strategy="epoch",

    save_strategy="no",

    fp16=True,

    report_to="none"
)


# ============================================================
# 3. Trainer
# ============================================================

welfake_vanilla_trainer = Trainer(

    model=welfake_vanilla_model,

    args=welfake_vanilla_args,

    train_dataset=welfake_train_dataset,

    eval_dataset=welfake_validation_dataset
)


# ============================================================
# 4. Train model
# ============================================================

print("\nTraining Vanilla RoBERTa on WELFake...")

welfake_vanilla_trainer.train()


# ============================================================
# 5. Evaluate on Test Dataset (Final Accuracy)
# ============================================================

welfake_test_predictions = welfake_vanilla_trainer.predict(
    welfake_test_dataset
)

welfake_test_preds = np.argmax(
    welfake_test_predictions.predictions,
    axis=1
)

welfake_test_accuracy = accuracy_score(
    welfake_test_predictions.label_ids,
    welfake_test_preds
)

print("\nVanilla RoBERTa WELFake Test Accuracy:", welfake_test_accuracy)


# ============================================================
# 6. Classification Report
# ============================================================

print("\nClassification Report:")

print(
    classification_report(
        welfake_test_predictions.label_ids,
        welfake_test_preds,
        target_names=["Fake", "Real"]
    )
)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Training Vanilla RoBERTa on WELFake...


Epoch,Training Loss,Validation Loss
1,0.013700,0.005842
2,0.006900,0.006600
3,0.003800,0.006417



Vanilla RoBERTa WELFake Test Accuracy: 0.9990752727945256

Classification Report:
              precision    recall  f1-score   support

        Fake       1.00      1.00      1.00      5254
        Real       1.00      1.00      1.00      5560

    accuracy                           1.00     10814
   macro avg       1.00      1.00      1.00     10814
weighted avg       1.00      1.00      1.00     10814



#  ISOT with train-test-val dataset 

In [ ]:
# ============================================================
# CELL FUNCTION:
# Proper ISOT split (Train / Validation / Test) + Pretraining + Evaluation
# ============================================================

import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer
)

# ============================================================
# 1. Load ISOT Dataset
# Download datasets from the links provided in README.md
# and update the paths below accordingly
# ============================================================

ISOT_PATH = "/path/to/your/dataset/"

fake_df = pd.read_csv(ISOT_PATH + "Fake.csv")
real_df = pd.read_csv(ISOT_PATH + "True.csv")

fake_df["label"] = 0
real_df["label"] = 1

isot_df = pd.concat([fake_df, real_df], ignore_index=True)

# Keep only required columns and clean
isot_df = isot_df[["text", "label"]]
isot_df.dropna(inplace=True)
isot_df.drop_duplicates(inplace=True)

print("Total ISOT samples:", len(isot_df))

# ============================================================
# 2. Proper Train / Validation / Test split
# ============================================================

# First split → Train (80%) and Temp (20%)
isot_train_df, isot_temp_df = train_test_split(
    isot_df,
    test_size=0.2,
    stratify=isot_df["label"],
    random_state=42
)

# Split Temp → Validation (10%) and Test (10%)
isot_validation_df, isot_test_df = train_test_split(
    isot_temp_df,
    test_size=0.5,
    stratify=isot_temp_df["label"],
    random_state=42
)

print("\nSplit sizes:")
print("Train:", len(isot_train_df))
print("Validation:", len(isot_validation_df))
print("Test:", len(isot_test_df))

# ============================================================
# 3. Load tokenizer
# ============================================================

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

# ============================================================
# 4. Convert to HuggingFace Dataset
# ============================================================

isot_train_dataset = Dataset.from_pandas(isot_train_df)
isot_validation_dataset = Dataset.from_pandas(isot_validation_df)
isot_test_dataset = Dataset.from_pandas(isot_test_df)

# Tokenize
isot_train_dataset = isot_train_dataset.map(tokenize_function, batched=True)
isot_validation_dataset = isot_validation_dataset.map(tokenize_function, batched=True)
isot_test_dataset = isot_test_dataset.map(tokenize_function, batched=True)

# Set format
isot_train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

isot_validation_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

isot_test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

# ============================================================
# 5. Load RoBERTa model
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

isot_model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
).to(device)

# ============================================================
# 6. Metrics function
# ============================================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc
    }

# ============================================================
# 7. Training Arguments
# ============================================================

isot_training_args = TrainingArguments(

    output_dir="./isot_proper_pretraining",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    report_to="none"
)

# ============================================================
# 8. Trainer
# ============================================================

isot_trainer = Trainer(

    model=isot_model,
    args=isot_training_args,

    train_dataset=isot_train_dataset,

    eval_dataset=isot_validation_dataset,

    compute_metrics=compute_metrics
)

# ============================================================
# 9. Train model
# ============================================================

print("\nStarting ISOT pretraining...")

isot_trainer.train()

# ============================================================
# 10. Evaluate on Validation set
# ============================================================

print("\nValidation Results:")

validation_results = isot_trainer.evaluate(
    eval_dataset=isot_validation_dataset
)

print(validation_results)

# ============================================================
# 11. Evaluate on Test set (final unbiased evaluation)
# ============================================================

print("\nTest Results:")

test_predictions = isot_trainer.predict(
    isot_test_dataset
)

test_preds = np.argmax(test_predictions.predictions, axis=1)

test_accuracy = accuracy_score(
    test_predictions.label_ids,
    test_preds
)

print("\nISOT Test Accuracy:", test_accuracy)

print("\nClassification Report:")
print(
    classification_report(
        test_predictions.label_ids,
        test_preds,
        target_names=["Fake", "Real"]
    )
)

# ============================================================
# 12. Save final model checkpoint
# ============================================================

isot_trainer.save_model("./isot_proper_pretrained_checkpoint")

print("\nISOT model saved successfully.")

Total ISOT samples: 38647

Split sizes:
Train: 30917
Validation: 3865
Test: 3865


Map:   0%|          | 0/30917 [00:00<?, ? examples/s]

Map:   0%|          | 0/3865 [00:00<?, ? examples/s]

Map:   0%|          | 0/3865 [00:00<?, ? examples/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Starting ISOT pretraining...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.002200,0.001938,0.999741
2,0.002200,0.005045,0.999483
3,0.001300,0.002923,0.999741



Validation Results:


{'eval_loss': 0.0019376389682292938, 'eval_accuracy': 0.9997412677878396, 'eval_runtime': 27.2042, 'eval_samples_per_second': 142.074, 'eval_steps_per_second': 8.896, 'epoch': 3.0}

Test Results:

ISOT Test Accuracy: 0.9994825355756791

Classification Report:
              precision    recall  f1-score   support

        Fake       1.00      1.00      1.00      1746
        Real       1.00      1.00      1.00      2119

    accuracy                           1.00      3865
   macro avg       1.00      1.00      1.00      3865
weighted avg       1.00      1.00      1.00      3865


ISOT model saved successfully.


In [15]:
import os
print(os.listdir("./isot_proper_pretraining"))

['checkpoint-5799', 'checkpoint-1933', 'checkpoint-3866']
